## LLM Validation

In [ ]:
# --- ENVIRONMENT CHECK ---
import sys
import subprocess

print(f"Python version: {sys.version.split()[0]}")

# --- REQUIRED PACKAGES ---
required = [
    "pandas", "numpy", "scikit-learn", "openpyxl", "tabulate", "python-docx",
]

for pkg in required:
    import_name = {"scikit-learn": "sklearn", "python-docx": "docx"}.get(pkg, pkg)
    try:
        __import__(import_name)
    except ImportError:
        try:
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", "-q", pkg],
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL,
            )
        except subprocess.CalledProcessError:
            print(f"Warning: could not install '{pkg}'. Install it manually.")

# --- IMPORTS ---
import math
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import cohen_kappa_score
from IPython.display import display, Markdown
from docx import Document
from docx.enum.section import WD_ORIENT
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.shared import Inches

Python version: 3.13.6


## 1. LLM test-retest reliability: three repeated runs on 200 abstracts

Sample 200 abstracts from the gold standard file and run the LLM classifier three times at low temperature. Compute pairwise Cohen's κ to assess labelling consistency across runs.

In [5]:
import pandas as pd
from pathlib import Path

infile = Path("../table/gold_standard_30_march.xlsx")
outfile = Path("../table/gold_standard_random_200.xlsx")
outfile.parent.mkdir(parents=True, exist_ok=True)

df200 = pd.read_excel(infile, sheet_name="in").copy()

if "id" not in df200.columns:
    if "scopus_id" in df200.columns:
        df200["id"] = df200["scopus_id"].astype(str).str.strip()
    else:
        df200["id"] = df200.index.astype(str)

df200 = df200.sample(n=200, random_state=42).copy()
df200.to_excel(outfile, sheet_name="in", index=False)

print(f"Saved: {outfile}")
print(f"N = {len(df200)}")


Saved: ../table/gold_standard_random_200.xlsx
N = 200


In [6]:
import pandas as pd
from pathlib import Path
from sklearn.metrics import cohen_kappa_score

RUN_DIR = Path("../concordance/concordance_outputs")

run1_path = RUN_DIR / "run_rerun1_temp0.1_2026-04-02_222144.csv"
run2_path = RUN_DIR / "run_rerun2_temp0.1_2026-04-02_222354.csv"
run3_path = RUN_DIR / "run_rerun3_temp0.1_2026-04-02_222534.csv"

def load_run(path, label_name):
    df = pd.read_csv(path).copy()
    df[label_name] = df["llm_label"].astype(str).str.strip().str.upper().map({"YES": 1, "NO": 0})
    return df[[label_name]].reset_index(drop=True)

def compare_runs(df_a, col_a, df_b, col_b):
    paired = pd.concat([df_a, df_b], axis=1).dropna()
    kappa = cohen_kappa_score(paired[col_a], paired[col_b])
    agreement = (paired[col_a] == paired[col_b]).mean() * 100
    return {
        "comparison": f"{col_a} vs {col_b}",
        "N": len(paired),
        "kappa": round(kappa, 3),
        "agreement_pct": round(agreement, 1),
    }

run1 = load_run(run1_path, "run1")
run2 = load_run(run2_path, "run2")
run3 = load_run(run3_path, "run3")

results = pd.DataFrame([
    compare_runs(run1, "run1", run2, "run2"),
    compare_runs(run1, "run1", run3, "run3"),
    compare_runs(run2, "run2", run3, "run3"),
])

display(results)

for _, r in results.iterrows():
    print(f"{r['comparison']}: κ={r['kappa']:.3f}, N={r['N']}, agreement={r['agreement_pct']:.1f}%")


,comparison,N,kappa,agreement_pct
0,run1 vs run2,200,0.905,98.0
1,run1 vs run3,200,0.930,98.5
2,run2 vs run3,200,0.978,99.5


run1 vs run2: κ=0.905, N=200, agreement=98.0%
run1 vs run3: κ=0.930, N=200, agreement=98.5%
run2 vs run3: κ=0.978, N=200, agreement=99.5%


## 2. LLM agreement with manual reviewers (gold standard)

Compare LLM labels against multiple human reviewers using the gold standard file. Reports Cohen's κ, % agreement, sensitivity, and specificity treating the LLM label as reference.

In [7]:
import pandas as pd
import numpy as np
from sklearn.metrics import cohen_kappa_score
from IPython.display import display, Markdown

# ----------------------------
# SETTINGS
# ----------------------------
file_path = "../table/gold_standard_30_march.xlsx"
sheet_name = "in"

reference_cols = ["llm_policy_claim"]

compare_cols = [
    "agreed_gold_standard",
    "agreed_gold_standard_with_exclusions",
    "DB review",
    "EC review",
    "MW review",
    "db re-review",
    "EC re-review",
]

display_labels = {
    "agreed_gold_standard": "agreed_gold_standard",
    "agreed_gold_standard_with_exclusions": "agreed_gold_standard_with_exclusions",
    "DB review": "DB review",
    "EC review": "EC review",
    "MW review": "MW review",
    "db re-review": "DB re-review",
    "EC re-review": "EC re-review",
}

# ----------------------------
# HELPER: recode binary values
# ----------------------------
na_strings = {"n/a", "na", "nan", "#n/a", "none", "", "-", "."}

def recode_binary(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s in na_strings:
        return np.nan
    yes_vals = {"1", "yes", "y", "true"}
    no_vals = {"0", "no", "n", "false"}
    if s in yes_vals:
        return 1
    if s in no_vals:
        return 0
    try:
        f = float(s)
        if f == 1:
            return 1
        if f == 0:
            return 0
    except:
        pass
    return np.nan

# ----------------------------
# LOAD DATA
# ----------------------------
df = pd.read_excel(file_path, sheet_name=sheet_name)

all_cols = reference_cols + compare_cols
missing_cols = []
for col in all_cols:
    if col not in df.columns:
        missing_cols.append(col)
    else:
        df[col + "_bin"] = df[col].apply(recode_binary)

def compare_to_reference(df, reference_col, other_col):
    ref_bin = reference_col + "_bin"
    oth_bin = other_col + "_bin"
    sub = df[[ref_bin, oth_bin]].dropna().copy()
    sub = sub[sub[ref_bin].isin([0, 1]) & sub[oth_bin].isin([0, 1])]
    label_other = display_labels.get(other_col, other_col)
    if len(sub) == 0:
        return {"comparison": f"{reference_col} vs {label_other}", "n": 0,
                "agreement_pct": np.nan, "kappa": np.nan,
                "sensitivity": np.nan, "specificity": np.nan}
    sub[ref_bin] = sub[ref_bin].astype(int)
    sub[oth_bin] = sub[oth_bin].astype(int)
    agreement = (sub[ref_bin] == sub[oth_bin]).mean() * 100
    kappa = cohen_kappa_score(sub[ref_bin], sub[oth_bin])
    tp = ((sub[ref_bin] == 1) & (sub[oth_bin] == 1)).sum()
    tn = ((sub[ref_bin] == 0) & (sub[oth_bin] == 0)).sum()
    fp = ((sub[ref_bin] == 0) & (sub[oth_bin] == 1)).sum()
    fn = ((sub[ref_bin] == 1) & (sub[oth_bin] == 0)).sum()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    return {
        "comparison": f"{reference_col} vs {label_other}",
        "n": len(sub),
        "agreement_pct": round(agreement, 1),
        "kappa": round(kappa, 3),
        "sensitivity": round(sensitivity, 3) if pd.notna(sensitivity) else np.nan,
        "specificity": round(specificity, 3) if pd.notna(specificity) else np.nan,
    }

# ----------------------------
# RUN COMPARISONS
# ----------------------------
md = ""
if missing_cols:
    md += "**Missing columns:**\n"
    for col in missing_cols:
        md += f"- `{col}`\n"
    md += "\n"

for ref_col in reference_cols:
    if ref_col in missing_cols:
        continue
    results = []
    for col in compare_cols:
        if col not in missing_cols and col in df.columns:
            results.append(compare_to_reference(df, ref_col, col))
    if results:
        results_df = pd.DataFrame(results)
        md += results_df.to_markdown(index=False)
    else:
        md += "_No valid comparisons available._"
    md += "\n\n"

display(Markdown(md))

| comparison                                               |   n |   agreement_pct |   kappa |   sensitivity |   specificity |
|:---------------------------------------------------------|----:|----------------:|--------:|--------------:|--------------:|
| llm_policy_claim vs agreed_gold_standard                 | 204 |            92.6 |   0.803 |         0.768 |         0.986 |
| llm_policy_claim vs agreed_gold_standard_with_exclusions | 197 |            92.9 |   0.805 |         0.769 |         0.986 |
| llm_policy_claim vs DB review                            | 204 |            91.2 |   0.765 |         0.75  |         0.973 |
| llm_policy_claim vs EC review                            |  94 |            89.4 |   0.705 |         0.68  |         0.971 |
| llm_policy_claim vs MW review                            | 104 |            95.2 |   0.882 |         0.9   |         0.973 |
| llm_policy_claim vs DB re-review                         |   5 |            80   |   0.545 |         1     |         0.5   |
| llm_policy_claim vs EC re-review                         | 204 |            92.6 |   0.803 |         0.768 |         0.986 |



In [8]:
sub = df[["DB review_bin", "EC re-review_bin"]].dropna().copy()
sub = sub[sub["DB review_bin"].isin([0, 1]) & sub["EC re-review_bin"].isin([0, 1])]

agreement = (sub["DB review_bin"] == sub["EC re-review_bin"]).mean() * 100
kappa = cohen_kappa_score(sub["DB review_bin"], sub["EC re-review_bin"])

print(f"DB vs EC: κ={kappa:.3f}, N={len(sub)}, agreement={agreement:.1f}%")


DB vs EC: κ=0.901, N=204, agreement=96.6%


## 3. Stratified 400-abstract manual validation

Load the stratified sample of 400 manually reviewed abstracts (DB + EC), match back to the main analytic dataset by Scopus ID / DOI, adjudicate discordant pairs using DB double-check, and compare LLM vs manual claim rates by time period. Outputs Supplementary Table 8.

In [44]:
import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------------
# 1. Load files
# ------------------------------------------------------------------
review_path = Path("../table/supp_stratified_sample_400_blinded_db.xlsx")
main_path = Path("../data/analysis/analysis_dataset_enriched_v2.csv")

review_df = pd.read_excel(review_path).copy()
main_df = pd.read_csv(main_path).copy()

print(f"Review file rows: {len(review_df)}")
print(f"Main analytic rows: {len(main_df)}")
print(f"Main dataset year range: {main_df['publication_year'].min()}–{main_df['publication_year'].max()}")

# ------------------------------------------------------------------
# 2. Normalise keys / labels
# ------------------------------------------------------------------
def norm_text(x):
    if pd.isna(x):
        return pd.NA
    s = str(x).strip()
    return s if s else pd.NA

def norm_doi(x):
    if pd.isna(x):
        return pd.NA
    s = str(x).strip().lower()
    s = s.replace("https://doi.org/", "").replace("http://doi.org/", "")
    return s if s else pd.NA

def norm_bool(x):
    if pd.isna(x):
        return pd.NA
    if isinstance(x, bool):
        return x
    s = str(x).strip().lower()
    if s in {"1", "1.0", "true", "t", "yes", "y"}:
        return True
    if s in {"0", "0.0", "false", "f", "no", "n"}:
        return False
    return pd.NA

main_df["scopus_id"] = main_df["scopus_id"].apply(norm_text)
main_df["doi"] = main_df["doi"].apply(norm_doi)

review_df["scopus_id"] = review_df["scopus_id"].apply(norm_text)
review_df["doi"] = review_df["doi"].apply(norm_doi)

for col in ["DB_policy_claim", "EC_policy_claim", "db_double_checked"]:
    if col in review_df.columns:
        review_df[col] = review_df[col].apply(norm_bool)

main_df["llm_policy_claim"] = main_df["llm_policy_claim"].apply(norm_bool)
main_df["publication_year"] = pd.to_numeric(main_df["publication_year"], errors="coerce").astype("Int64")

# ------------------------------------------------------------------
# 3. Build lookup tables
# ------------------------------------------------------------------
lookup_scopus = (
    main_df[["scopus_id", "publication_year", "llm_policy_claim"]]
    .dropna(subset=["scopus_id"])
    .drop_duplicates(subset=["scopus_id"])
    .copy()
)

lookup_doi = (
    main_df[["doi", "publication_year", "llm_policy_claim"]]
    .dropna(subset=["doi"])
    .drop_duplicates(subset=["doi"])
    .copy()
)

# ------------------------------------------------------------------
# 4. Match back to main dataset
# ------------------------------------------------------------------
review_merged = review_df.merge(
    lookup_scopus,
    on="scopus_id",
    how="left",
    validate="many_to_one"
)
review_merged["matched_by"] = np.where(review_merged["publication_year"].notna(), "scopus_id", pd.NA)

missing_mask = review_merged["publication_year"].isna()

if missing_mask.any():
    doi_fill = review_merged.loc[missing_mask, ["doi"]].merge(
        lookup_doi,
        on="doi",
        how="left",
        validate="many_to_one"
    )
    review_merged.loc[missing_mask, "publication_year"] = doi_fill["publication_year"].values
    review_merged.loc[missing_mask, "llm_policy_claim"] = doi_fill["llm_policy_claim"].values
    review_merged.loc[
        missing_mask & doi_fill["publication_year"].notna().values, "matched_by"
    ] = "doi"

# ------------------------------------------------------------------
# 5. Final manual label
# ------------------------------------------------------------------
# If DB and EC agree, use that; if discordant, use db_double_checked as adjudication.
review_merged["manual_claim_final"] = np.where(
    review_merged["DB_policy_claim"] == review_merged["EC_policy_claim"],
    review_merged["DB_policy_claim"],
    review_merged["db_double_checked"]
)
review_merged["manual_claim_final"] = pd.Series(review_merged["manual_claim_final"]).astype("boolean")

# ------------------------------------------------------------------
# 6. Checks
# ------------------------------------------------------------------
n_total = len(review_merged)
n_matched = int(review_merged["publication_year"].notna().sum())
n_unmatched = int(review_merged["publication_year"].isna().sum())
n_discordant = int((review_merged["DB_policy_claim"] != review_merged["EC_policy_claim"]).fillna(False).sum())
n_final_nonmissing = int(review_merged["manual_claim_final"].notna().sum())

print(f"\nManual review file rows: {n_total}")
print(f"Matched back to analytic sample: {n_matched}")
print(f"Unmatched rows: {n_unmatched}")
print(f"DB/EC discordant rows: {n_discordant}")
print(f"Rows with final manual label: {n_final_nonmissing}")
print("\nMatch method counts:")
print(review_merged["matched_by"].value_counts(dropna=False))

if n_unmatched:
    print("\nUnmatched rows preview:")
    display(review_merged.loc[
        review_merged["publication_year"].isna(),
        ["review_id", "scopus_id", "doi", "title"]
    ].head(20))

# ------------------------------------------------------------------
# 7. Overall summary
# ------------------------------------------------------------------
matched_review = review_merged[review_merged["publication_year"].notna()].copy()

n_manual_claim = int(matched_review["manual_claim_final"].fillna(False).sum())
manual_claim_rate = 100 * n_manual_claim / len(matched_review) if len(matched_review) else np.nan

n_llm_claim = int(matched_review["llm_policy_claim"].fillna(False).sum())
llm_claim_rate = 100 * n_llm_claim / len(matched_review) if len(matched_review) else np.nan

overall_summary = pd.DataFrame([{
    "n_reviewed": n_total,
    "n_matched_to_main_df": len(matched_review),
    "n_db_ec_discordant": n_discordant,
    "n_manual_policy_claim": n_manual_claim,
    "manual_policy_claim_rate_pct": round(manual_claim_rate, 1) if pd.notna(manual_claim_rate) else np.nan,
    "n_llm_policy_claim": n_llm_claim,
    "llm_policy_claim_rate_pct": round(llm_claim_rate, 1) if pd.notna(llm_claim_rate) else np.nan,
}])

display(overall_summary)

# ------------------------------------------------------------------
# 8. Period summaries
# ------------------------------------------------------------------
periods = {
    "1990-1999": (1990, 2000),
    "2000-2009": (2000, 2010),
    "2010-2019": (2010, 2020),
    "2020-2024": (2020, 2025),
}

rows = []
for label, (start, end) in periods.items():
    full_sub = main_df[
        (main_df["publication_year"] >= start) &
        (main_df["publication_year"] < end)
    ].copy()

    review_sub = matched_review[
        (matched_review["publication_year"] >= start) &
        (matched_review["publication_year"] < end)
    ].copy()

    n_full = len(full_sub)
    n_review = len(review_sub)

    full_rate = 100 * full_sub["llm_policy_claim"].astype(bool).mean() if n_full else np.nan
    sample_llm_rate = 100 * review_sub["llm_policy_claim"].astype(bool).mean() if n_review else np.nan
    manual_rate = 100 * review_sub["manual_claim_final"].astype(bool).mean() if n_review else np.nan

    rows.append({
        "period": label,
        "n_full": n_full,
        "full_llm_claim_rate_pct": round(full_rate, 1) if pd.notna(full_rate) else np.nan,
        "n_review_sample": n_review,
        "sample_llm_claim_rate_pct": round(sample_llm_rate, 1) if pd.notna(sample_llm_rate) else np.nan,
        "manual_claim_rate_pct": round(manual_rate, 1) if pd.notna(manual_rate) else np.nan,
    })

period_summary = pd.DataFrame(rows)
display(period_summary)

Review file rows: 400
Main analytic rows: 45807
Main dataset year range: 1990–2024

Manual review file rows: 400
Matched back to analytic sample: 400
Unmatched rows: 0
DB/EC discordant rows: 24
Rows with final manual label: 400

Match method counts:
matched_by
scopus_id    400
Name: count, dtype: int64


,n_reviewed,n_matched_to_main_df,n_db_ec_discordant,n_manual_policy_claim,manual_policy_claim_rate_pct,n_llm_policy_claim,llm_policy_claim_rate_pct
0,400,400,24,127,31.8,97,24.2


,period,n_full,full_llm_claim_rate_pct,n_review_sample,sample_llm_claim_rate_pct,manual_claim_rate_pct
0,1990-1999,10436,17.6,114,13.2,19.3
1,2000-2009,12529,22.8,115,22.6,27.8
2,2010-2019,15464,28.4,114,27.2,36.0
3,2020-2024,7378,35.8,57,43.9,56.1


## Supplementary Table: Policy claim prevalence by period (400 manually classified abstracts)

Computes the proportion of abstracts judged to contain a policy claim in each time period, with Wilson 95% CIs and a chi-square test across periods. Exports as CSV and Word document (supp_table8_manual_400_by_period).

*Note: these period-level estimates are plotted in notebook 9 (9_main_analyses_supplemental).*

In [45]:
import math
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import chi2 as chi2_dist
from docx import Document
from docx.enum.section import WD_ORIENT
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.shared import Inches

supp_periods = {
    "1990-1999": (1990, 2000),
    "2000-2009": (2000, 2010),
    "2010-2019": (2010, 2020),
    "2020-2024": (2020, 2025),
}

def wilson_ci(successes, total, z=1.959963984540054):
    if total == 0:
        return (np.nan, np.nan)
    p = successes / total
    denom = 1 + (z ** 2) / total
    center = (p + (z ** 2) / (2 * total)) / denom
    half = z * math.sqrt((p * (1 - p) + (z ** 2) / (4 * total)) / total) / denom
    return center - half, center + half

def chi_square_pvalue_2xk(successes, totals):
    successes = np.asarray(successes, dtype=float)
    totals = np.asarray(totals, dtype=float)
    failures = totals - successes

    grand_total = totals.sum()
    grand_successes = successes.sum()
    grand_failures = failures.sum()

    expected_successes = totals * grand_successes / grand_total
    expected_failures = totals * grand_failures / grand_total

    chi2_stat = (((successes - expected_successes) ** 2) / expected_successes).sum()
    chi2_stat += (((failures - expected_failures) ** 2) / expected_failures).sum()

    df = len(totals) - 1
    p_value = chi2_dist.sf(chi2_stat, df)
    return chi2_stat, p_value

def format_pvalue(p_value):
    return "<0.001" if p_value < 0.001 else f"{p_value:.3f}"

manual_period_rows = []
successes = []
totals = []

for label, (start, end) in supp_periods.items():
    sub = matched_review[
        (matched_review["publication_year"] >= start)
        & (matched_review["publication_year"] < end)
        & matched_review["manual_claim_final"].notna()
    ].copy()

    total = len(sub)
    success = int(sub["manual_claim_final"].astype(int).sum()) if total else 0
    estimate = 100 * success / total if total else np.nan
    ci_low, ci_high = wilson_ci(success, total)

    successes.append(success)
    totals.append(total)

    manual_period_rows.append({
        "Period": label,
        "Policy claims, n/N": f"{success}/{total}",
        "Estimate (%)": round(estimate, 1) if pd.notna(estimate) else np.nan,
        "95% CI": f"({100 * ci_low:.1f}, {100 * ci_high:.1f})",
    })

chi2_stat, overall_p = chi_square_pvalue_2xk(successes, totals)
supp_table_manual_400 = pd.DataFrame(manual_period_rows)
supp_table_manual_400["P for overall difference"] = ""
supp_table_manual_400.loc[0, "P for overall difference"] = format_pvalue(overall_p)

display(supp_table_manual_400)

first_est = supp_table_manual_400.loc[
    supp_table_manual_400["Period"] == "1990-1999", "Estimate (%)"
].iloc[0]
last_est = supp_table_manual_400.loc[
    supp_table_manual_400["Period"] == "2020-2024", "Estimate (%)"
].iloc[0]

print(
    f"An increasing trend was also found in the 400 abstracts manually classified "
    f"(from {first_est:.1f}% in 1990-1999 to {last_est:.1f}% in 2020-2024; Supplementary Table X)."
)
print(f"Chi-square test across periods: chi2={chi2_stat:.2f}, p={format_pvalue(overall_p)}")

out_dir = Path("../table")
out_dir.mkdir(parents=True, exist_ok=True)

csv_path = out_dir / "supp_table8_manual_400_by_period.csv"
docx_path = out_dir / "supp_table8_manual_400_by_period.docx"
supp_table_manual_400.to_csv(csv_path, index=False)

doc = Document()
section = doc.sections[0]
section.orientation = WD_ORIENT.PORTRAIT
if section.page_width > section.page_height:
    section.page_width, section.page_height = section.page_height, section.page_width

doc.add_heading(
    "Supplementary Table 8. Policy claim prevalence across time periods in the 400 manually classified abstracts",
    level=1,
)
doc.add_paragraph(
    "Estimate = proportion of manually classified abstracts in each period judged to contain a policy claim. "
    "95% CIs are Wilson confidence intervals. P value is from a Pearson chi-square test comparing the binary outcome across periods."
)

cols = supp_table_manual_400.columns.tolist()
table = doc.add_table(rows=1, cols=len(cols), style="Table Grid")
table.alignment = WD_TABLE_ALIGNMENT.CENTER

for j, header in enumerate(cols):
    cell = table.rows[0].cells[j]
    p = cell.paragraphs[0]
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    run = p.add_run(str(header))
    run.bold = True

for _, row in supp_table_manual_400.iterrows():
    cells = table.add_row().cells
    for j, col in enumerate(cols):
        val = row[col]
        txt = "" if pd.isna(val) else str(val)
        p = cells[j].paragraphs[0]
        p.alignment = WD_ALIGN_PARAGRAPH.LEFT if j == 0 else WD_ALIGN_PARAGRAPH.RIGHT
        p.add_run(txt)

table.columns[0].width = Inches(1.35)
table.columns[1].width = Inches(1.45)
table.columns[2].width = Inches(1.05)
table.columns[3].width = Inches(1.55)
table.columns[4].width = Inches(1.35)

doc.save(docx_path)
print("Saved CSV: ../table/supp_table8_manual_400_by_period.csv")
print("Saved DOCX: ../table/supp_table8_manual_400_by_period.docx")

,Period,"Policy claims, n/N",Estimate (%),95% CI,P for overall difference
0,1990-1999,22/114,19.3,"(13.1, 27.5)",<0.001
1,2000-2009,32/115,27.8,"(20.5, 36.6)",
2,2010-2019,41/114,36.0,"(27.7, 45.1)",
3,2020-2024,32/57,56.1,"(43.3, 68.2)",


An increasing trend was also found in the 400 abstracts manually classified (from 19.3% in 1990-1999 to 56.1% in 2020-2024; Supplementary Table X).
Chi-square test across periods: chi2=25.56, p=<0.001
Saved CSV: ../table/supp_table8_manual_400_by_period.csv
Saved DOCX: ../table/supp_table8_manual_400_by_period.docx


## validate study design 

create dataset to validate study design

In [9]:
!python 6_make_review_sample.py design-stratified --seed 42

Exported 400 rows to: table/manual_review_by_design_50_each.csv

Counts by design in exported sample:
design_combined
Case-control                50
Cohort                      50
Cross-sectional             50
Ecological / Time-series    50
Experimental                50
Other/None                  50
Qualitative                 50
Quasi-experimental          50


In [46]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import cohen_kappa_score

review_path = Path("/Users/wangmengyao/Desktop/Github/policyclaims/table/manual_review_by_design_50_each_EC.xlsx")
df = pd.read_excel(review_path).copy()
df.columns = df.columns.astype(str).str.strip()

MANUAL_COL = "manual_check1"
AUTO_COLS = [
    "design_strict",
    "design_keywords",
    "design_combined",
    "design_combined_unambiguous",
]

def clean_design(x):
    if pd.isna(x):
        return pd.NA
    s = str(x).strip().lower()
    if s in {"", "na", "n/a", "nan"}:
        return pd.NA

    mapping = {
        "experimental": "Experimental",
        "quasi-experimental": "Quasi-experimental",
        "quasi experimental": "Quasi-experimental",
        "cohort": "Cohort",
        "case-control": "Case-control",
        "case control": "Case-control",
        "cross-sectional": "Cross-sectional",
        "cross sectional": "Cross-sectional",
        "qualitative": "Qualitative",
        "ecological": "Ecological / Time-series",
        "time-series": "Ecological / Time-series",
        "time series": "Ecological / Time-series",
        "ecological / time-series": "Ecological / Time-series",
        "other/none": "Other/None",
        "other / none": "Other/None",
        "none": "Other/None",
    }
    return mapping.get(s, s)

for col in [MANUAL_COL] + AUTO_COLS:
    if col in df.columns:
        df[col + "_clean"] = df[col].apply(clean_design)

def compare_designs(df_in, manual_col, auto_col):
    sub = df_in[[manual_col, auto_col]].dropna().copy()
    if len(sub) == 0:
        return {
            "comparison": f"{auto_col} vs {manual_col}",
            "n": 0,
            "agreement_pct": np.nan,
            "kappa": np.nan,
        }

    agreement = (sub[manual_col] == sub[auto_col]).mean() * 100
    kappa = cohen_kappa_score(sub[manual_col], sub[auto_col])

    return {
        "comparison": f"{auto_col} vs {manual_col}",
        "n": len(sub),
        "agreement_pct": round(agreement, 1),
        "kappa": round(kappa, 3),
    }

results = pd.DataFrame([
    compare_designs(df, MANUAL_COL + "_clean", col + "_clean")
    for col in AUTO_COLS
    if col in df.columns
]).sort_values(["kappa", "agreement_pct"], ascending=False).reset_index(drop=True)

display(results)

# 1) likely lack of data: algorithm says Other/None, Emilie gives a concrete design
lack_of_data = df[
    df[MANUAL_COL + "_clean"].notna() &
    df["design_combined_clean"].notna() &
    (df["design_combined_clean"] == "Other/None") &
    (df[MANUAL_COL + "_clean"] != "Other/None") &
    (df[MANUAL_COL + "_clean"] != df["design_combined_clean"])
][[
    "review_id", "title", "design_combined", MANUAL_COL, "Notes EC"
]].copy()

# 2) likely actual mistakes: algorithm gives a concrete non-Other label, but differs from Emilie
actual_mistakes = df[
    df[MANUAL_COL + "_clean"].notna() &
    df["design_combined_clean"].notna() &
    (df["design_combined_clean"] != "Other/None") &
    (df[MANUAL_COL + "_clean"] != df["design_combined_clean"])
][[
    "review_id", "title", "design_strict", "design_keywords", "design_combined",
    MANUAL_COL, "Notes EC"
]].copy()

print(f"Lack-of-data cases (design_combined = Other/None, Emilie assigned a design): {len(lack_of_data)}")
display(lack_of_data)

print(f"Potential actual mistakes (design_combined assigned a concrete but different design): {len(actual_mistakes)}")
display(actual_mistakes)

# 3) Emilie-noted cases
ec_noted = df[df["Notes EC"].notna() & (df["Notes EC"].astype(str).str.strip() != "")][[
    "review_id", "title", "design_strict", "design_keywords", "design_combined",
    MANUAL_COL, "Notes EC"
]].copy()

print(f"Cases with Emilie notes: {len(ec_noted)}")
display(ec_noted)


,comparison,n,agreement_pct,kappa
0,design_combined_clean vs manual_check1_clean,400,81.8,0.793
1,design_strict_clean vs manual_check1_clean,400,78.5,0.758
2,design_combined_unambiguous_clean vs manual_ch...,400,78.0,0.752
3,design_keywords_clean vs manual_check1_clean,400,18.0,0.129


Lack-of-data cases (design_combined = Other/None, Emilie assigned a design): 39


,review_id,title,design_combined,manual_check1,Notes EC
8,DES-009,Association between economic fluctuations and ...,Other/None,cohort,longitudinal study
23,DES-024,The association between discrimination and hea...,Other/None,cross-sectional,NaN
28,DES-029,Early Onset of Distress Disorders and High-Sch...,Other/None,cohort,longitudinal study
32,DES-033,Job stress and cardiovascular risk factors in ...,Other/None,cross-sectional,NaN
61,DES-062,Epidemiology of sudden cardiac death in Camero...,Other/None,cross-sectional,NaN
62,DES-063,"Social support, exposure to violence and trans...",Other/None,cross-sectional,NaN
67,DES-068,Longitudinal effects of perinatal social suppo...,Other/None,cohort,longitudinal study
79,DES-080,Misclassification rates for current smokers mi...,Other/None,cross-sectional,NaN
83,DES-084,Nonresponse and intensity of follow-up in an e...,Other/None,cross-sectional,NaN
84,DES-085,The epidemiology of melioidosis in Ubon Ratcha...,Other/None,cohort,NaN


Potential actual mistakes (design_combined assigned a concrete but different design): 34


,review_id,title,design_strict,design_keywords,design_combined,manual_check1,Notes EC
38,DES-039,American cutaneous leishmaniasis in Southeast ...,Ecological / Time-series,Other/None,Ecological / Time-series,ecological study,NaN
45,DES-046,Health for Hearts United Longitudinal Trial: I...,Quasi-experimental,Other/None,Quasi-experimental,experimental,NaN
57,DES-058,Television exposure is related to fear of avia...,Ecological / Time-series,Ecological / Time-series,Ecological / Time-series,ecological study,NaN
59,DES-060,Deaths in collective dwellings and inequalitie...,Ecological / Time-series,Other/None,Ecological / Time-series,ecological study,NaN
94,DES-095,Cancer mortality and exposure to chemical carc...,Ecological / Time-series,Ecological / Time-series,Ecological / Time-series,ecological study,NaN
104,DES-105,An analysis of the process of human immunodefi...,Ecological / Time-series,Cohort,Ecological / Time-series,ecological study,NaN
111,DES-112,An evaluation of the impact of a large reducti...,Quasi-experimental,Other/None,Quasi-experimental,times series,NaN
120,DES-121,The impact of the Great Recession on Californi...,Ecological / Time-series,Other/None,Ecological / Time-series,times series,NaN
125,DES-126,Evaluating a campaign to detect early stage br...,Ecological / Time-series,Other/None,Ecological / Time-series,times series,NaN
133,DES-134,Short term respiratory health effects of ambie...,Ecological / Time-series,Other/None,Ecological / Time-series,times series,NaN


Cases with Emilie notes: 55


,review_id,title,design_strict,design_keywords,design_combined,manual_check1,Notes EC
2,DES-003,Differential misclassification and the assessm...,Case-control,Case-control,Case-control,case-control,They mention limitations with case-control stu...
7,DES-008,A longitudinal study evaluating adverse childh...,Cohort,Other/None,Cohort,cohort,longitudinal study
8,DES-009,Association between economic fluctuations and ...,Other/None,Other/None,Other/None,cohort,longitudinal study
12,DES-013,Association of Chlamydia trachomatis with pers...,Cohort,Other/None,Cohort,cohort,longitudinal study
13,DES-014,Time trend analysis of social inequalities in ...,Cohort,Other/None,Cohort,cohort,longitudinal study
16,DES-017,Bringing “The Real Cost” to Life Through Break...,Other/None,Other/None,Other/None,other/none,unclear whether this is an empirical paper
17,DES-018,Effectiveness of the cigarette ignition propen...,Quasi-experimental,Other/None,Quasi-experimental,quasi-experimental,"ITS, should be time series?"
19,DES-020,Medicaid expansion initiative in Massachusetts...,Quasi-experimental,Other/None,Quasi-experimental,quasi-experimental,"ITS, should be time series?"
20,DES-021,Scarring in Utero: An Attempt to Validate with...,Cohort,Other/None,Cohort,cohort,longitudinal study
27,DES-028,Comparing like with like: Some historical mile...,Experimental,Other/None,Experimental,experimental,unclear whether this is an empirical paper or ...


In [47]:
import pandas as pd
import importlib.util
from pathlib import Path

# ------------------------------------------------------------
# Load current v2 dataset
# ------------------------------------------------------------
main_path = Path("../data/analysis/analysis_dataset_enriched_v2.csv")
review_path = Path("../table/manual_review_by_design_50_each_EC.xlsx")

main_df = pd.read_csv(main_path).copy()
review_df = pd.read_excel(review_path).copy()

main_df["scopus_id"] = main_df["scopus_id"].astype(str).str.strip()
review_df["scopus_id"] = review_df["scopus_id"].astype(str).str.strip()

# ------------------------------------------------------------
# Load current pattern definitions from 5_
# ------------------------------------------------------------
script_path = Path("../code/5d_add_study_design_and_topics.py")
spec = importlib.util.spec_from_file_location("study_design_module", script_path)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

TITLE_ABSTRACT_PATTERNS = mod.TITLE_ABSTRACT_PATTERNS
KEYWORD_DESIGN_PATTERNS = mod.KEYWORD_DESIGN_PATTERNS
normalize_text = mod.normalize_text

# ------------------------------------------------------------
# Helper to show which patterns match
# ------------------------------------------------------------
def find_matches(title, abstract, keywords):
    text = normalize_text(f"{title} {abstract}")
    kw = normalize_text(str(keywords))

    title_abs_hits = {}
    keyword_hits = {}

    for design, pats in TITLE_ABSTRACT_PATTERNS.items():
        hits = [p for p in pats if p in text]
        if hits:
            title_abs_hits[design] = hits

    for design, pats in KEYWORD_DESIGN_PATTERNS.items():
        hits = [p for p in pats if p in kw]
        if hits:
            keyword_hits[design] = hits

    return title_abs_hits, keyword_hits

def inspect_review_id(review_id):
    row_review = review_df.loc[review_df["review_id"] == review_id].copy()
    if row_review.empty:
        print(f"{review_id} not found in review file.")
        return

    row_review = row_review.iloc[0]
    scopus_id = str(row_review["scopus_id"]).strip()

    row_main = main_df.loc[main_df["scopus_id"] == scopus_id].copy()
    if row_main.empty:
        print(f"{review_id} / {scopus_id} not found in main v2 dataset.")
        return

    row_main = row_main.iloc[0]

    print("=" * 100)
    print(f"Review ID: {review_id}")
    print(f"Title: {row_main['title']}")
    print(f"Scopus ID: {scopus_id}")
    print()
    print("Current v2 classifications:")
    for col in ["design_strict", "design_keywords", "design_combined", "design_combined_unambiguous"]:
        if col in row_main.index:
            print(f"  {col}: {row_main[col]}")
    print()
    print("Emilie review:")
    print(f"  manual_check1: {row_review.get('manual_check1')}")
    print(f"  Notes EC: {row_review.get('Notes EC')}")
    print()

    ta_hits, kw_hits = find_matches(
        row_main.get("title", ""),
        row_main.get("abstract", ""),
        row_main.get("keywords", ""),
    )

    print("Title/abstract pattern hits:")
    if ta_hits:
        for design, hits in ta_hits.items():
            print(f"  {design}: {hits}")
    else:
        print("  None")

    print("\nKeyword pattern hits:")
    if kw_hits:
        for design, hits in kw_hits.items():
            print(f"  {design}: {hits}")
    else:
        print("  None")

    print("\nAbstract preview:")
    print(str(row_main.get("abstract", ""))[:1200])
    print()

# inspect_review_id("DES-112")
# inspect_review_id("DES-121")
# inspect_review_id("DES-009")
# inspect_review_id("DES-046")
inspect_review_id("DES-062")
#inspect_review_id("DES-063")


Review ID: DES-062
Title: Epidemiology of sudden cardiac death in Cameroon: The first population-based cohort survey in sub-Saharan Africa
Scopus ID: SCOPUS_ID:85030458185

Current v2 classifications:
  design_strict: Other/None
  design_keywords: Other/None
  design_combined: Other/None
  design_combined_unambiguous: Other/None

Emilie review:
  manual_check1: cross-sectional
  Notes EC: nan

Title/abstract pattern hits:
  None

Keyword pattern hits:
  None

Abstract preview:
Background: Incidence estimates of sudden cardiac death (SCD) in sub-Saharan Africa (SSA) are unknown. Method: Over 12 months, the household administrative office and health community committee within neighbourhoods in two health areas of Douala, Cameroon, registered all deaths among 86 188 inhabitants aged > 18 years. As part of an extended multi-source surveillance system, the Emergency Medical Service (EMS), local medical examiners and district hospital mortuaries were also surveyed. Whereas two physicians inv

In [48]:
import pandas as pd
import numpy as np

# Assumes df is already loaded from manual_review_by_design_50_each_EC.xlsx
# and these cleaned columns already exist:
# - manual_check1_clean
# - design_combined_clean
# - design_strict
# - design_keywords
# - design_combined
# - Notes EC

work = df.copy()

def norm_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip().lower()

work["notes_ec_clean"] = work["Notes EC"].apply(norm_text)
work["manual_raw_clean"] = work["manual_check1"].apply(norm_text)
work["combined_raw_clean"] = work["design_combined"].apply(norm_text)
work["title_clean"] = work["title"].apply(norm_text)
work["abstract_clean"] = work["abstract"].apply(norm_text)

mismatch = work[
    work["manual_check1_clean"].notna() &
    work["design_combined_clean"].notna() &
    (work["manual_check1_clean"] != work["design_combined_clean"])
].copy()

def classify_mismatch(row):
    manual = row["manual_check1_clean"]
    auto = row["design_combined_clean"]
    notes = row["notes_ec_clean"]
    title = row["title_clean"]
    abstract = row["abstract_clean"]
    full_text = f"{title} {abstract}"

    # 1. LLM says Other/None, Emilie gives a concrete design
    if auto == "Other/None" and manual != "Other/None":
        if any(term in notes for term in [
            "important discrepancy",
            "rct",
            "time series",
            "case-control",
            "cross-sectional",
            "cohort",
            "experimental",
            "quasi-experimental"
        ]):
            return "likely_fixable_other_none"
        return "likely_lack_of_data"

    # 2. LLM gives a concrete design, Emilie says none/other, and notes suggest review/methods/non-empirical
    if manual == "Other/None" and auto != "Other/None":
        if any(term in notes for term in [
            "not an empirical paper",
            "literature review",
            "systematic review",
            "review of",
            "not empirical",
            "unclear whether this is an empirical paper",
            "new method",
            "monte carlo simulation",
            "cost-effectiveness",
        ]):
            return "likely_non_empirical_false_positive"
        return "likely_conceptual_disagreement"

    # 3. Time-series / ITS conflicts
    if any(term in notes for term in [
        "time series",
        "times series",
        "time-series",
        "its",
        "pre-post",
        "pre post",
    ]):
        return "likely_time_series_conflict"

    if ("time series" in full_text or "interrupted time series" in full_text) and (
        auto in {"Quasi-experimental", "Ecological / Time-series"} or
        manual in {"Quasi-experimental", "Ecological / Time-series"}
    ):
        return "likely_time_series_conflict"

    # 4. Experimental / trial-related mismatches
    if any(term in notes for term in [
        "rct",
        "trial",
        "randomized",
        "randomised",
    ]):
        return "likely_experimental_conflict"

    # 5. Longitudinal / cohort ambiguity
    if any(term in notes for term in [
        "longitudinal study",
        "longitudinal",
    ]):
        return "likely_longitudinal_ambiguity"

    # 6. Explicit important discrepancy but not otherwise caught
    if "important discrepancy" in notes:
        return "likely_fixable_actual_mistake"

    return "other_mismatch"

mismatch["mismatch_type"] = mismatch.apply(classify_mismatch, axis=1)

summary = (
    mismatch["mismatch_type"]
    .value_counts(dropna=False)
    .rename_axis("mismatch_type")
    .reset_index(name="n")
)

display(summary)

cols_to_show = [
    "review_id",
    "title",
    "design_strict",
    "design_keywords",
    "design_combined",
    "manual_check1",
    "Notes EC",
    "mismatch_type",
]

display(
    mismatch[cols_to_show]
    .sort_values(["mismatch_type", "review_id"])
    .reset_index(drop=True)
)


,mismatch_type,n
0,likely_lack_of_data,37
1,other_mismatch,15
2,likely_fixable_actual_mistake,7
3,likely_non_empirical_false_positive,5
4,likely_time_series_conflict,3
5,likely_experimental_conflict,3
6,likely_fixable_other_none,2
7,likely_conceptual_disagreement,1


,review_id,title,design_strict,design_keywords,design_combined,manual_check1,Notes EC,mismatch_type
0,DES-360,Efficiency loss from categorizing quantitative...,Qualitative,Case-control,Qualitative,none,unclear whether this is an empirical study,likely_conceptual_disagreement
1,DES-259,Association of serum lipoproteins and health-r...,Experimental,Other/None,Experimental,cross-sectional,"this is an important discrepancy, trials are m...",likely_experimental_conflict
2,DES-320,Evaluation of community-wide interventions: Th...,Experimental,Other/None,Experimental,ecological study,"important discrepancy, they mention that they ...",likely_experimental_conflict
3,DES-332,Loneliness and its cross-sectional association...,Experimental,Other/None,Experimental,cross-sectional,"important discrepancy, that's an odd one: ther...",likely_experimental_conflict
4,DES-139,Effect of sunscreen and clothing on the number...,Experimental,Other/None,Experimental,cross-sectional,this is an important discrepancy,likely_fixable_actual_mistake
...,...,...,...,...,...,...,...,...
68,DES-226,Temperature and cardiovascular mortality in Ri...,Other/None,Ecological / Time-series,Ecological / Time-series,ecological study,NaN,other_mismatch
69,DES-253,Urban Containment Policies and Physical Activi...,Ecological / Time-series,Other/None,Ecological / Time-series,ecological study,NaN,other_mismatch
70,DES-333,"Indicators of deprivation, voting patterns, an...",Ecological / Time-series,Other/None,Ecological / Time-series,ecological study,NaN,other_mismatch
71,DES-368,Bayesian factor analysis to calculate a depriv...,Ecological / Time-series,Other/None,Ecological / Time-series,ecological study,NaN,other_mismatch


In [49]:
priority = mismatch[
    mismatch["mismatch_type"].isin([
        "likely_time_series_conflict",
        "likely_non_empirical_false_positive",
        "likely_experimental_conflict",
        "likely_fixable_actual_mistake",
        "likely_fixable_other_none",
    ])
].copy()

display(
    priority[
        ["review_id", "title", "design_strict", "design_keywords", "design_combined",
         "manual_check1", "Notes EC", "mismatch_type"]
    ].sort_values(["mismatch_type", "review_id"]).reset_index(drop=True)
)


,review_id,title,design_strict,design_keywords,design_combined,manual_check1,Notes EC,mismatch_type
0,DES-259,Association of serum lipoproteins and health-r...,Experimental,Other/None,Experimental,cross-sectional,"this is an important discrepancy, trials are m...",likely_experimental_conflict
1,DES-320,Evaluation of community-wide interventions: Th...,Experimental,Other/None,Experimental,ecological study,"important discrepancy, they mention that they ...",likely_experimental_conflict
2,DES-332,Loneliness and its cross-sectional association...,Experimental,Other/None,Experimental,cross-sectional,"important discrepancy, that's an odd one: ther...",likely_experimental_conflict
3,DES-139,Effect of sunscreen and clothing on the number...,Experimental,Other/None,Experimental,cross-sectional,this is an important discrepancy,likely_fixable_actual_mistake
4,DES-202,Public perceptions of cardiovascular risk fact...,Other/None,Experimental,Experimental,cross-sectional,this is an important discrepancy,likely_fixable_actual_mistake
5,DES-204,The aging population in Sweden: Can declining ...,Other/None,Experimental,Experimental,time series,this is an important discrepancy,likely_fixable_actual_mistake
6,DES-232,Are manual workers at higher risk of death tha...,Other/None,Experimental,Experimental,cohort,this is an important discrepancy,likely_fixable_actual_mistake
7,DES-238,The association between fast-food outlet proxi...,Quasi-experimental,Other/None,Quasi-experimental,cross-sectional,this is an important discrepancy,likely_fixable_actual_mistake
8,DES-316,Risk factors for coronary heart disease in Afr...,Other/None,Experimental,Experimental,case-control,this is an important discrepancy,likely_fixable_actual_mistake
9,DES-391,Infant feeding and mental and motor developmen...,Experimental,Other/None,Experimental,cohort,important discrepancy,likely_fixable_actual_mistake


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import cohen_kappa_score

# ------------------------------------------------------------
# 1. Load old review sample + new v2e study design outputs
# ------------------------------------------------------------
main_path = Path("../data/analysis/analysis_dataset_enriched_v2.csv")
review_path = Path("../table/manual_review_by_design_50_each_EC.xlsx")

main_df = pd.read_csv(main_path).copy()
review_df = pd.read_excel(review_path).copy()

main_df["scopus_id"] = main_df["scopus_id"].astype(str).str.strip()
review_df["scopus_id"] = review_df["scopus_id"].astype(str).str.strip()

design_cols = [
    "scopus_id",
    "design_strict",
    "design_keywords",
    "design_combined",
    "design_combined_unambiguous",
]

df = review_df.merge(
    main_df[design_cols],
    on="scopus_id",
    how="left",
).copy()

# review file already contains the old design outputs; merged columns are the new ones
df = df.rename(columns={
    "design_strict_x": "design_strict_old",
    "design_keywords_x": "design_keywords_old",
    "design_combined_x": "design_combined_old",
    "design_combined_unambiguous_x": "design_combined_unambiguous_old",
    "design_strict_y": "design_strict_new",
    "design_keywords_y": "design_keywords_new",
    "design_combined_y": "design_combined_new",
    "design_combined_unambiguous_y": "design_combined_unambiguous_new",
})

print(f"Loaded {len(review_df)} reviewed records from {review_path.name}")
print(f"Loaded updated design dataset from {main_path.name}")
print("Missing new design_combined after merge:", int(df["design_combined_new"].isna().sum()))

# ------------------------------------------------------------
# 2. Clean labels
# ------------------------------------------------------------
def clean_design(x):
    if pd.isna(x):
        return pd.NA
    s = str(x).strip().lower()
    if s in {"", "na", "n/a", "nan"}:
        return pd.NA

    mapping = {
        "experimental": "Experimental",
        "quasi-experimental": "Quasi-experimental",
        "quasi experimental": "Quasi-experimental",
        "cohort": "Cohort",
        "case-control": "Case-control",
        "case control": "Case-control",
        "cross-sectional": "Cross-sectional",
        "cross sectional": "Cross-sectional",
        "qualitative": "Qualitative",
        "ecological": "Ecological / Time-series",
        "ecological study": "Ecological / Time-series",
        "ecologic study": "Ecological / Time-series",
        "time-series": "Ecological / Time-series",
        "time series": "Ecological / Time-series",
        "times series": "Ecological / Time-series",
        "ecological / time-series": "Ecological / Time-series",
        "other/none": "Other/None",
        "other / none": "Other/None",
        "none": "Other/None",
        "other": "Other/None",
    }
    return mapping.get(s, s)

df_eval = df.copy()

for col in [
    "manual_check1",
    "design_strict_old",
    "design_keywords_old",
    "design_combined_old",
    "design_combined_unambiguous_old",
    "design_strict_new",
    "design_keywords_new",
    "design_combined_new",
    "design_combined_unambiguous_new",
]:
    if col in df_eval.columns:
        df_eval[col + "_clean"] = df_eval[col].apply(clean_design)

# ------------------------------------------------------------
# 3. Compare old vs new against Emilie manual review
# ------------------------------------------------------------
def compare_designs(df_in, manual_col, auto_col):
    sub = df_in[[manual_col, auto_col]].dropna().copy()
    if len(sub) == 0:
        return {
            "comparison": f"{auto_col} vs {manual_col}",
            "n": 0,
            "agreement_pct": np.nan,
            "kappa": np.nan,
        }

    agreement = (sub[manual_col] == sub[auto_col]).mean() * 100
    kappa = cohen_kappa_score(sub[manual_col], sub[auto_col])

    return {
        "comparison": f"{auto_col} vs {manual_col}",
        "n": len(sub),
        "agreement_pct": round(agreement, 1),
        "kappa": round(kappa, 3),
    }

results = pd.DataFrame([
    compare_designs(df_eval, "manual_check1_clean", "design_strict_old_clean"),
    compare_designs(df_eval, "manual_check1_clean", "design_combined_old_clean"),
    compare_designs(df_eval, "manual_check1_clean", "design_combined_unambiguous_old_clean"),
    compare_designs(df_eval, "manual_check1_clean", "design_strict_new_clean"),
    compare_designs(df_eval, "manual_check1_clean", "design_combined_new_clean"),
    compare_designs(df_eval, "manual_check1_clean", "design_combined_unambiguous_new_clean"),
    compare_designs(df_eval, "manual_check1_clean", "design_keywords_new_clean"),
]).sort_values(["kappa", "agreement_pct"], ascending=False).reset_index(drop=True)

display(results)

# ------------------------------------------------------------
# 4. Show only records where old and new combined differ
# ------------------------------------------------------------
changed = df_eval[
    df_eval["design_combined_old_clean"] != df_eval["design_combined_new_clean"]
][[
    "review_id",
    "title",
    "manual_check1_clean",
    "design_combined_old_clean",
    "design_combined_new_clean",
    "Notes EC",
]].copy()

changed["old_correct"] = (
    changed["design_combined_old_clean"] == changed["manual_check1_clean"]
)
changed["new_correct"] = (
    changed["design_combined_new_clean"] == changed["manual_check1_clean"]
)

print("Number of records changed by 5f:", len(changed))
display(changed)

change_summary = changed.groupby(["old_correct", "new_correct"]).size().reset_index(name="n")
display(change_summary)

# ------------------------------------------------------------
# 5. Quick interpretation
# ------------------------------------------------------------
old_row = results.loc[results["comparison"] == "design_combined_old_clean vs manual_check1_clean"].iloc[0]
new_row = results.loc[results["comparison"] == "design_combined_new_clean vs manual_check1_clean"].iloc[0]

print("\nSummary:")
print(
    f"Old combined: agreement {old_row['agreement_pct']}%, kappa {old_row['kappa']}"
)
print(
    f"New combined (5e): agreement {new_row['agreement_pct']}%, kappa {new_row['kappa']}"
)

if new_row["kappa"] > old_row["kappa"]:
    print("5e improves validation performance over the old version.")
elif new_row["kappa"] < old_row["kappa"]:
    print("5e reduces validation performance relative to the old version.")
else:
    print("5e and the old version have identical kappa.")


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import cohen_kappa_score

review_path = Path("../table/manual_review_by_design_50_each_EC.xlsx")
review_df = pd.read_excel(review_path).copy()
review_df["scopus_id"] = review_df["scopus_id"].astype(str).str.strip()

version_files = {
    "old_reviewfile": None,  # use labels already stored in the review Excel
    "5": "../data/analysis/analysis_dataset_enriched_v2.csv",
    "5b": "../data/analysis/analysis_dataset_enriched_v2b.csv",
    "5c": "../data/analysis/analysis_dataset_enriched_v2c.csv",
    "5d": "../data/analysis/analysis_dataset_enriched_v2d.csv",
    "5e": "../data/analysis/analysis_dataset_enriched_v2e.csv",
    "5f": "../data/analysis/analysis_dataset_enriched_v2f.csv",
}

def clean_design(x):
    if pd.isna(x):
        return pd.NA
    s = str(x).strip().lower()
    if s in {"", "na", "n/a", "nan"}:
        return pd.NA
    mapping = {
        "experimental": "Experimental",
        "quasi-experimental": "Quasi-experimental",
        "quasi experimental": "Quasi-experimental",
        "cohort": "Cohort",
        "case-control": "Case-control",
        "case control": "Case-control",
        "cross-sectional": "Cross-sectional",
        "cross sectional": "Cross-sectional",
        "qualitative": "Qualitative",
        "ecological": "Ecological / Time-series",
        "ecological study": "Ecological / Time-series",
        "ecologic study": "Ecological / Time-series",
        "time-series": "Ecological / Time-series",
        "time series": "Ecological / Time-series",
        "times series": "Ecological / Time-series",
        "ecological / time-series": "Ecological / Time-series",
        "other/none": "Other/None",
        "other / none": "Other/None",
        "none": "Other/None",
        "other": "Other/None",
    }
    return mapping.get(s, s)

def compare_designs(df_in, manual_col, auto_col):
    sub = df_in[[manual_col, auto_col]].dropna().copy()
    agreement = (sub[manual_col] == sub[auto_col]).mean() * 100
    kappa = cohen_kappa_score(sub[manual_col], sub[auto_col])
    return {
        "comparison": f"{auto_col} vs {manual_col}",
        "n": len(sub),
        "agreement_pct": round(agreement, 1),
        "kappa": round(kappa, 3),
    }

# manual labels
base = review_df.copy()
base["manual_check1_clean"] = base["manual_check1"].apply(clean_design)

results = []

# old labels stored in review file
old_df = base.copy()
for col in ["design_strict", "design_keywords", "design_combined", "design_combined_unambiguous"]:
    old_df[f"{col}_clean"] = old_df[col].apply(clean_design)

results.extend([
    {"version": "old_reviewfile", **compare_designs(old_df, "manual_check1_clean", "design_strict_clean")},
    {"version": "old_reviewfile", **compare_designs(old_df, "manual_check1_clean", "design_combined_clean")},
    {"version": "old_reviewfile", **compare_designs(old_df, "manual_check1_clean", "design_combined_unambiguous_clean")},
    {"version": "old_reviewfile", **compare_designs(old_df, "manual_check1_clean", "design_keywords_clean")},
])

# rerun versions
design_cols = [
    "scopus_id",
    "design_strict",
    "design_keywords",
    "design_combined",
    "design_combined_unambiguous",
]

for version, path_str in version_files.items():
    if version == "old_reviewfile":
        continue

    path = Path(path_str)
    main_df = pd.read_csv(path).copy()
    main_df["scopus_id"] = main_df["scopus_id"].astype(str).str.strip()

    merged = base.merge(main_df[design_cols], on="scopus_id", how="left")
    for col in ["design_strict", "design_keywords", "design_combined", "design_combined_unambiguous"]:
        merged[f"{col}_clean"] = merged[col].apply(clean_design)

    results.extend([
        {"version": version, **compare_designs(merged, "manual_check1_clean", "design_strict_clean")},
        {"version": version, **compare_designs(merged, "manual_check1_clean", "design_combined_clean")},
        {"version": version, **compare_designs(merged, "manual_check1_clean", "design_combined_unambiguous_clean")},
        {"version": version, **compare_designs(merged, "manual_check1_clean", "design_keywords_clean")},
    ])

results_df = pd.DataFrame(results)

# most useful summary: combined only
combined_summary = (
    results_df[results_df["comparison"].str.contains("design_combined_clean")]
    .sort_values(["kappa", "agreement_pct"], ascending=False)
    .reset_index(drop=True)
)

display(combined_summary)
display(results_df.sort_values(["version", "comparison"]).reset_index(drop=True))